# 📝 Text Summarization with Hugging Face Transformers
### End-to-End NLP Pipeline — Google Colab Edition

This notebook runs the complete **Text Summarization** project pipeline:
1. **Environment Setup** — Install dependencies, mount Drive (optional)
2. **Project Structure** — Create directories, config files, source modules
3. **Stage 1 — Data Ingestion** — Download & extract SAMSum dataset
4. **Stage 2 — Data Transformation** — Tokenize with PEGASUS tokenizer
5. **Stage 3 — Model Training** — Fine-tune `google/pegasus-cnn_dailymail`
6. **Stage 4 — Model Evaluation** — Compute ROUGE scores
7. **Prediction / Demo** — Summarize custom dialogue
8. **Groq LLM Demo** — Fast inference via `llama-3.1-8b-instant` on Groq
9. **Output Packaging** — Zip all artifacts for download

> **Prerequisites:** Add your `GROQ_API_KEY` to Colab Secrets  
> *(Left sidebar → 🔑 Secrets → Add `GROQ_API_KEY`)*


---
## ⚙️ Phase 0 — Environment Setup

Install all required packages. GPU runtime is strongly recommended for training.  
*Runtime → Change runtime type → T4 GPU*


In [1]:
# ── Install all dependencies ──────────────────────────────────────────────────
# transformers[sentencepiece] includes SentencePiece needed by PEGASUS tokenizer
# evaluate provides the ROUGE metric used in evaluation
# groq is the official Groq Python SDK for fast LLM inference
# python-box provides dot-access config dicts; ensure adds type annotations

import subprocess, sys

packages = [
    "transformers[sentencepiece]",
    "datasets",
    "sacrebleu",
    "rouge_score",
    "py7zr",
    "pandas",
    "nltk",
    "tqdm",
    "PyYAML",
    "matplotlib",
    "torch",
    "python-box==6.0.2",
    "evaluate",
    "groq",          # Groq SDK for LLaMA fast inference
    "accelerate",    # required by recent HF Trainer
]

print("Installing packages … (this may take 2-3 minutes)")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✅ All packages installed successfully.")


Installing packages … (this may take 2-3 minutes)
✅ All packages installed successfully.


In [2]:
!pip install transformers -q

In [3]:
# ── Verify GPU availability ───────────────────────────────────────────────────
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Training will be very slow on CPU.")
    print("    Go to Runtime → Change runtime type → GPU (T4)")


Using device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


---
## 🗂️ Phase 1 — Recreate Project Structure

We write out all source modules inline so the notebook is fully self-contained
(no `git clone` required). The layout mirrors the original package structure:

```
/content/TextSummarizer/
├── config/config.yaml
├── params.yaml
├── src/textSummarizer/
│   ├── constants/
│   ├── entity/
│   ├── logging/
│   ├── utils/
│   ├── components/
│   ├── config/
│   └── pipeline/
└── artifacts/   (created at runtime)
```


In [4]:
# ── Set working directory ─────────────────────────────────────────────────────
import os

PROJECT_ROOT = "/content/TextSummarizer"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")


Working directory: /content/TextSummarizer


In [5]:
# ── Helper: write a file, creating parent directories as needed ───────────────
from pathlib import Path

def write_file(path: str, content: str):
    """Write content to path, auto-creating any missing parent directories."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    print(f"  created: {path}")


In [6]:
# ── 1-A  config/config.yaml ───────────────────────────────────────────────────
# Central configuration: paths for all pipeline stages and model checkpoints.

write_file("config/config.yaml", """
artifacts_root: artifacts

data_ingestion:
  root_dir: artifacts/data_ingestion
  source_URL: https://github.com/krishnaik06/datasets/raw/refs/heads/main/summarizer-data.zip
  local_data_file: artifacts/data_ingestion/data.zip
  unzip_dir: artifacts/data_ingestion

data_transformation:
  root_dir: artifacts/data_transformation
  data_path: artifacts/data_ingestion/samsum_dataset
  tokenizer_name: google/pegasus-cnn_dailymail

model_trainer:
  root_dir: artifacts/model_trainer
  data_path: artifacts/data_transformation/samsum_dataset
  model_ckpt: google/pegasus-cnn_dailymail

model_evaluation:
  root_dir: artifacts/model_evaluation
  data_path: artifacts/data_transformation/samsum_dataset
  model_path: artifacts/model_trainer/pegasus-samsum-model
  tokenizer_path: artifacts/model_trainer/tokenizer
  metric_file_name: artifacts/model_evaluation/metrics.csv
""")

# ── 1-B  params.yaml ──────────────────────────────────────────────────────────
# HuggingFace TrainingArguments hyper-parameters, kept separate so they can
# be tuned without touching the code.

write_file("params.yaml", """
TrainingArguments:
  num_train_epochs: 1
  warmup_steps: 500
  per_device_train_batch_size: 1
  weight_decay: 0.01
  logging_steps: 10
  evaluation_strategy: steps
  eval_steps: 500
  save_steps: 1000000
  gradient_accumulation_steps: 32 # Increased from 16 to 32
""")

print("\n✅ Config files written.")

  created: config/config.yaml
  created: params.yaml

✅ Config files written.


In [7]:
# ── 1-C  Package __init__ stubs ───────────────────────────────────────────────
# Python requires __init__.py in every package directory.

init_dirs = [
    "src/__init__.py",
    "src/textSummarizer/__init__.py",
    "src/textSummarizer/components/__init__.py",
    "src/textSummarizer/config/__init__.py",
    "src/textSummarizer/constants/__init__.py",
    "src/textSummarizer/entity/__init__stub.py",  # will be overwritten below
    "src/textSummarizer/logging/__init__stub.py",
    "src/textSummarizer/pipeline/__init__.py",
    "src/textSummarizer/utils/__init__.py",
]

for f in init_dirs:
    if f.endswith("stub.py"):
        continue  # handled individually below
    write_file(f, "")

print("✅ Package stubs created.")


  created: src/__init__.py
  created: src/textSummarizer/__init__.py
  created: src/textSummarizer/components/__init__.py
  created: src/textSummarizer/config/__init__.py
  created: src/textSummarizer/constants/__init__.py
  created: src/textSummarizer/pipeline/__init__.py
  created: src/textSummarizer/utils/__init__.py
✅ Package stubs created.


In [8]:
# ── 1-D  constants/__init__.py ────────────────────────────────────────────────
# Defines the canonical paths to the two YAML config files.

write_file("src/textSummarizer/constants/__init__.py", """\
from pathlib import Path

# Paths to project configuration files (relative to project root)
CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
""")


  created: src/textSummarizer/constants/__init__.py


In [9]:
write_file("src/textSummarizer/entity/__init__.py", '''\
from dataclasses import dataclass
from pathlib import Path


@dataclass
class DataIngestionConfig:
    """Config for downloading and extracting the raw dataset."""
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


@dataclass
class DataTransformationConfig:
    """Config for tokenizing the dataset and saving to disk."""
    root_dir: Path
    data_path: Path
    tokenizer_name: str          # HuggingFace model ID for tokenizer


@dataclass
class ModelTrainerConfig:
    """Config for fine-tuning the seq2seq model."""
    root_dir: Path
    data_path: Path
    model_ckpt: str              # HuggingFace model checkpoint to fine-tune
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int


@dataclass(frozen=True)
class ModelEvaluationConfig:
    """Config for ROUGE evaluation of the fine-tuned model (immutable)."""
    root_dir: Path
    data_path: Path
    model_path: Path             # path to the saved fine-tuned model
    tokenizer_path: Path         # path to the saved tokenizer
    metric_file_name: Path       # output CSV for ROUGE scores
''')

  created: src/textSummarizer/entity/__init__.py


In [10]:
# ── 1-F  logging/__init__.py ──────────────────────────────────────────────────
# Centralised logger: writes to both a rotating log file and stdout.

write_file("src/textSummarizer/logging/__init__.py", """\
import os
import sys
import logging

# Directory and format for log files
LOG_DIR = "logs"
LOG_FORMAT = "[%(asctime)s: %(levelname)s: %(module)s: %(message)s]"
LOG_FILE = os.path.join(LOG_DIR, "pipeline.log")

os.makedirs(LOG_DIR, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    handlers=[
        logging.FileHandler(LOG_FILE),    # persist to disk
        logging.StreamHandler(sys.stdout) # show in Colab cell output
    ]
)

# Named logger used across the entire project
logger = logging.getLogger("textSummarizer")
""")


  created: src/textSummarizer/logging/__init__.py


In [11]:
write_file("src/textSummarizer/utils/common.py", '''\
import os
from pathlib import Path
from typing import Any

import yaml
from box import ConfigBox
from box.exceptions import BoxValueError

from src.textSummarizer.logging import logger


def read_yaml(path_to_yaml: Path) -> ConfigBox:
    """
    Load a YAML file and return it as a ConfigBox for dot-access.
    """
    try:
        with open(path_to_yaml) as f:
            content = yaml.safe_load(f)
        logger.info(f"YAML loaded: {path_to_yaml}")
        return ConfigBox(content)
    except BoxValueError:
        raise ValueError(f"YAML file is empty: {path_to_yaml}")
    except Exception as e:
        raise e


def create_directories(path_to_directories: list, verbose: bool = True):
    """
    Create a list of directories, ignoring existing ones.
    """
    for path in path_to_directories:
        os.makedirs(path, exist_ok=True)
        if verbose:
            logger.info(f"Directory created: {path}")
''')

  created: src/textSummarizer/utils/common.py


In [12]:
write_file("src/textSummarizer/config/configuration.py", '''\
from src.textSummarizer.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.textSummarizer.utils.common import read_yaml, create_directories
from src.textSummarizer.entity import (
    DataIngestionConfig,
    DataTransformationConfig,
    ModelTrainerConfig,
    ModelEvaluationConfig,
)


class ConfigurationManager:
    """
    Reads YAML config files once and exposes typed getter methods
    for each pipeline stage's configuration object.
    """

    def __init__(
        self,
        config_path=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_filepath)   # fixed: was self.paramss

        # Ensure the top-level artifacts directory exists before any stage runs
        create_directories([self.config.artifacts_root])

    # ── Stage 1 ──────────────────────────────────────────────────────────────

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        """Return config for the data ingestion stage."""
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        return DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir,
        )

    # ── Stage 2 ──────────────────────────────────────────────────────────────

    def get_data_transformation_config(self) -> DataTransformationConfig:
        """Return config for the data transformation stage."""
        config = self.config.data_transformation
        create_directories([config.root_dir])

        return DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name,
        )

    # ── Stage 3 ──────────────────────────────────────────────────────────────

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        """Return config for the model training stage (merges config + params)."""
        config = self.config.model_trainer
        params = self.params.TrainingArguments
        create_directories([config.root_dir])

        return ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            evaluation_strategy=params.evaluation_strategy,
            eval_steps=params.eval_steps,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps,
        )

    # ── Stage 4 ──────────────────────────────────────────────────────────────

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        """Return config for the model evaluation stage."""
        config = self.config.model_evaluation
        create_directories([config.root_dir])

        return ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_name=config.metric_file_name,
        )
''')

  created: src/textSummarizer/config/configuration.py


In [21]:
write_file("src/textSummarizer/components/data_transformation.py", '''\
import os

from datasets import load_from_disk
from transformers import AutoTokenizer

from src.textSummarizer.entity import DataTransformationConfig
from src.textSummarizer.logging import logger


class DataTransformation:
    """
    Tokenizes the SAMSum dataset using the PEGASUS tokenizer and saves
    the tokenized dataset to disk in HuggingFace Arrow format.
    """

    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)
        logger.info(f"Tokenizer loaded: {config.tokenizer_name}")

    def convert_examples_to_features(self, example_batch: dict) -> dict:
        """
        Map function applied to each batch: tokenize source (dialogue) and
        target (summary). Labels use the tokenizer's target context manager
        for proper decoder input formatting.
        """
        input_encodings = self.tokenizer(
            example_batch["dialogue"], max_length=1024, truncation=True, padding="max_length"
        )

        # Tokenize labels directly without as_target_tokenizer
        target_encodings = self.tokenizer(
            example_batch["summary"], max_length=128, truncation=True, padding="max_length"
        )

        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"],
        }

    def convert(self):
        logger.info(f"Loading raw dataset from {self.config.data_path}")
        dataset_samsum = load_from_disk(self.config.data_path)

        logger.info("Tokenizing dataset (batched) …")
        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features, batched=True
        )

        save_path = os.path.join(self.config.root_dir, "samsum_dataset")
        dataset_samsum_pt.save_to_disk(save_path)
        logger.info(f"Tokenized dataset saved to {save_path}")
''')
print("✅ All components written.")

  created: src/textSummarizer/components/data_transformation.py
✅ All components written.


In [14]:
write_file("src/textSummarizer/components/data_ingestion.py", '''\
import os
import zipfile
import urllib.request as request

from src.textSummarizer.entity import DataIngestionConfig
from src.textSummarizer.logging import logger


class DataIngestion:
    """
    Handles downloading the raw dataset zip from a URL and extracting it.
    Skips the download if the zip already exists (idempotent).
    """

    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        """Download the zip file unless it already exists locally."""
        if not os.path.exists(self.config.local_data_file):
            logger.info(f"Downloading dataset from {self.config.source_URL} …")
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file,
            )
            logger.info(f"Downloaded → {filename}")
        else:
            logger.info(f"Dataset zip already exists, skipping download.")

    def extract_zip_file(self):
        """Extract the downloaded zip into the configured unzip directory."""
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        logger.info(f"Extracting zip to {unzip_path} …")
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info("Extraction complete.")
''')
print("✅ DataIngestion component written.")

  created: src/textSummarizer/components/data_ingestion.py
✅ DataIngestion component written.


In [15]:
write_file("src/textSummarizer/pipeline/stage_1_data_ingestion_pipeline.py", '''\
from src.textSummarizer.config.configuration import ConfigurationManager
from src.textSummarizer.components.data_ingestion import DataIngestion
from src.textSummarizer.logging import logger


class DataIngestionTrainingPipeline:
    """Orchestrates Stage 1: raw data download and extraction."""

    def __init__(self):
        pass

    def initiate_data_ingestion(self):
        logger.info("Stage 1 — Data Ingestion started.")
        config = ConfigurationManager()
        data_ingestion_config = config.get_data_ingestion_config()
        data_ingestion = DataIngestion(config=data_ingestion_config)
        data_ingestion.download_file()
        data_ingestion.extract_zip_file()
        logger.info("Stage 1 — Data Ingestion complete.")
''')


write_file("src/textSummarizer/pipeline/stage_2_data_transformation_pipeline.py", '''\
from src.textSummarizer.config.configuration import ConfigurationManager
from src.textSummarizer.components.data_transformation import DataTransformation
from src.textSummarizer.logging import logger


class DataTransformationTrainingPipeline:
    """Orchestrates Stage 2: tokenization and dataset serialisation."""

    def __init__(self):
        pass

    def initiate_data_transformation(self):
        logger.info("Stage 2 — Data Transformation started.")
        config = ConfigurationManager()
        data_transformation_config = config.get_data_transformation_config()
        data_transformation = DataTransformation(config=data_transformation_config)
        data_transformation.convert()
        logger.info("Stage 2 — Data Transformation complete.")
''')


write_file("src/textSummarizer/pipeline/stage_3_model_trainer_pipeline.py", '''\
from src.textSummarizer.config.configuration import ConfigurationManager
from src.textSummarizer.components.model_trainer import ModelTrainer
from src.textSummarizer.logging import logger


class ModelTrainerTrainingPipeline:
    """Orchestrates Stage 3: model fine-tuning and artifact saving."""

    def __init__(self):
        pass

    def initiate_model_trainer(self):
        logger.info("Stage 3 — Model Trainer started.")
        config = ConfigurationManager()
        model_trainer_config = config.get_model_trainer_config()
        model_trainer = ModelTrainer(config=model_trainer_config)
        model_trainer.train()
        logger.info("Stage 3 — Model Trainer complete.")
''')


write_file("src/textSummarizer/pipeline/stage_4_model_evaluation.py", '''\
from src.textSummarizer.config.configuration import ConfigurationManager
from src.textSummarizer.components.model_evaluation import ModelEvaluation
from src.textSummarizer.logging import logger


class ModelEvaluationTrainingPipeline:
    """Orchestrates Stage 4: ROUGE metric computation and CSV export."""

    def __init__(self):
        pass

    def initiate_model_evaluation(self):
        logger.info("Stage 4 — Model Evaluation started.")
        config = ConfigurationManager()
        model_evaluation_config = config.get_model_evaluation_config()
        model_evaluation = ModelEvaluation(config=model_evaluation_config)
        scores = model_evaluation.evaluate()
        logger.info("Stage 4 — Model Evaluation complete.")
        return scores
''')


write_file("src/textSummarizer/pipeline/prediction_pipeline.py", '''\
from transformers import AutoTokenizer, pipeline

from src.textSummarizer.config.configuration import ConfigurationManager
from src.textSummarizer.logging import logger


class PredictionPipeline:
    """
    Loads the fine-tuned model and tokenizer from disk and provides a
    single `predict()` method for inference on new dialogues.
    """

    def __init__(self):
        self.config = ConfigurationManager().get_model_evaluation_config()

    def predict(self, text: str) -> str:
        """
        Generate a summary for the provided dialogue text.

        Args:
            text (str): Raw dialogue to summarise.

        Returns:
            str: Model-generated summary.
        """
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)

        gen_kwargs = {"length_penalty": 0.8, "num_beams": 8, "max_length": 128}

        summarizer = pipeline(
            "summarization",
            model=self.config.model_path,
            tokenizer=tokenizer,
        )

        logger.info("Running prediction …")
        output = summarizer(text, **gen_kwargs)[0]["summary_text"]
        logger.info("Prediction complete.")
        return output
''')

print("✅ All pipeline stages written.")

  created: src/textSummarizer/pipeline/stage_1_data_ingestion_pipeline.py
  created: src/textSummarizer/pipeline/stage_2_data_transformation_pipeline.py
  created: src/textSummarizer/pipeline/stage_3_model_trainer_pipeline.py
  created: src/textSummarizer/pipeline/stage_4_model_evaluation.py
  created: src/textSummarizer/pipeline/prediction_pipeline.py
✅ All pipeline stages written.


In [16]:
# ── 1-K  Verify the full project structure ────────────────────────────────────
import subprocess
result = subprocess.run(["find", ".", "-name", "*.py", "-not", "-path", "./.git/*"],
                        capture_output=True, text=True)
print(result.stdout)
print("✅ Project structure verified.")


./src/textSummarizer/components/model_evaluation.py
./src/textSummarizer/components/data_ingestion.py
./src/textSummarizer/components/__init__.py
./src/textSummarizer/components/model_trainer.py
./src/textSummarizer/components/data_transformation.py
./src/textSummarizer/config/__init__.py
./src/textSummarizer/config/configuration.py
./src/textSummarizer/__init__.py
./src/textSummarizer/pipeline/stage_4_model_evaluation.py
./src/textSummarizer/pipeline/stage_2_data_transformation_pipeline.py
./src/textSummarizer/pipeline/prediction_pipeline.py
./src/textSummarizer/pipeline/__init__.py
./src/textSummarizer/pipeline/stage_3_model_trainer_pipeline.py
./src/textSummarizer/pipeline/stage_1_data_ingestion_pipeline.py
./src/textSummarizer/entity/__init__.py
./src/textSummarizer/logging/__init__.py
./src/textSummarizer/utils/__init__.py
./src/textSummarizer/utils/common.py
./src/textSummarizer/constants/__init__.py
./src/__init__.py

✅ Project structure verified.


---
## 📥 Phase 2 — Stage 1: Data Ingestion

Downloads the SAMSum dialogue dataset (≈ 3 MB) and extracts it into
`artifacts/data_ingestion/samsum_dataset/`.


In [17]:
# ── Run Stage 1: Data Ingestion ───────────────────────────────────────────────
# Adds the project root to sys.path so absolute imports (src.textSummarizer.*)
# resolve correctly throughout the notebook.

import sys
sys.path.insert(0, "/content/TextSummarizer")

from src.textSummarizer.pipeline.stage_1_data_ingestion_pipeline import (
    DataIngestionTrainingPipeline,
)

STAGE_NAME = "Data Ingestion"
try:
    print(f">>> Stage: {STAGE_NAME} <<<")
    pipeline_obj = DataIngestionTrainingPipeline()
    pipeline_obj.initiate_data_ingestion()
    print(f"✅ {STAGE_NAME} completed successfully.")
except Exception as e:
    print(f"❌ {STAGE_NAME} failed: {e}")
    raise

>>> Stage: Data Ingestion <<<
✅ Data Ingestion completed successfully.


In [18]:
# ── Quick sanity check: list extracted files ──────────────────────────────────
import os

data_dir = "artifacts/data_ingestion"
for root, dirs, files in os.walk(data_dir):
    level = root.replace(data_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:5]:   # show at most 5 files per dir
        print(f"{indent}  {f}")


data_ingestion/
  samsum-test.csv
  data.zip
  samsum-train.csv
  samsum-validation.csv
  samsum_dataset/
    dataset_dict.json
    train/
      dataset_info.json
      data-00000-of-00001.arrow
      cache-3bc6a7e4e296f7d9.arrow
      state.json
    validation/
      dataset_info.json
      data-00000-of-00001.arrow
      cache-629ab9ae32eb34f6.arrow
      state.json
    test/
      dataset_info.json
      cache-12e00b8818d57e36.arrow
      cache-4ba2a941c5e2358c.arrow
      data-00000-of-00001.arrow
      state.json


---
## 🔄 Phase 3 — Stage 2: Data Transformation

Tokenizes dialogues and summaries using the PEGASUS tokenizer.
Output is saved in HuggingFace Arrow format at
`artifacts/data_transformation/samsum_dataset/`.

> ⏱ Expect ~3–5 minutes (downloads tokenizer + processes ~16k examples).


In [23]:
# ── Run Stage 2: Data Transformation ─────────────────────────────────────────
import importlib
from src.textSummarizer.components import data_transformation
from src.textSummarizer.pipeline import stage_2_data_transformation_pipeline # Import the module

# Reload the component module first
importlib.reload(data_transformation)
# Then reload the pipeline module that uses the component
importlib.reload(stage_2_data_transformation_pipeline)

# Now, import the class from the reloaded pipeline module
from src.textSummarizer.pipeline.stage_2_data_transformation_pipeline import (
    DataTransformationTrainingPipeline,
)

STAGE_NAME = "Data Transformation"
try:
    print(f">>> Stage: {STAGE_NAME} <<<")
    pipeline_obj = DataTransformationTrainingPipeline()
    pipeline_obj.initiate_data_transformation()
    print(f"✅ {STAGE_NAME} completed successfully.")
except Exception as e:
    print(f"❌ {STAGE_NAME} failed: {e}")
    raise

>>> Stage: Data Transformation <<<


Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14732 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]

✅ Data Transformation completed successfully.


In [24]:
import os

import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq # Added DataCollatorForSeq2Seq

from src.textSummarizer.entity import ModelTrainerConfig
from src.textSummarizer.logging import logger


class ModelTrainer:
    """
    Fine-tunes a pre-trained sequence-to-sequence model (PEGASUS)
    on the tokenized SAMSum dataset using HuggingFace Trainer API.
    """

    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Load tokenizer and model
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)

        # Load tokenized dataset
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # Prepare TrainingArguments
        training_args = TrainingArguments(
            output_dir=self.config.root_dir, # directory for saving checkpoints
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            # Additional recommended arguments
            logging_dir=f"{self.config.root_dir}/logs",
            report_to="none", # Disable integrations with W&B, MLflow etc.
            push_to_hub=False,
        )

        # Define data collator to handle dynamic padding
        data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_pegasus)

        # Create Trainer instance
        trainer = Trainer(
            model=model_pegasus,
            args=training_args,
            tokenizer=tokenizer,          # Explicitly pass tokenizer
            data_collator=data_collator,  # Explicitly pass data collator
            train_dataset=dataset_samsum_pt["test"], # Using 'test' split for faster demo
            eval_dataset=dataset_samsum_pt["validation"],
        )

        logger.info("Starting model training (this may take a while) …")
        trainer.train()
        logger.info("Model training complete.")

        # Save model and tokenizer
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))
        logger.info(f"Fine-tuned model saved to {self.config.root_dir}/pegasus-samsum-model")
        logger.info(f"Tokenizer saved to {self.config.root_dir}/tokenizer")


---
## 🏋️ Phase 4 — Stage 3: Model Training

Fine-tunes `google/pegasus-cnn_dailymail` on the SAMSum dataset.

| Setting | Value |
|---------|-------|
| Epochs | 1 |
| Batch size | 1 |
| Gradient accumulation | 16 steps |
| Warmup steps | 500 |

> ⏱ On a T4 GPU: ~30–60 minutes for 1 epoch over the test split.  
> 💡 The training uses the `test` split (smaller) to reduce Colab runtime.


In [ ]:
# ── Run Stage 3: Model Training ───────────────────────────────────────────────
import importlib
from src.textSummarizer.components import model_trainer # Import the module
from src.textSummarizer.pipeline import stage_3_model_trainer_pipeline # Import the pipeline module

# Reload the component module first
importlib.reload(model_trainer)
# Then reload the pipeline module that uses the component
importlib.reload(stage_3_model_trainer_pipeline)

from src.textSummarizer.pipeline.stage_3_model_trainer_pipeline import (
    ModelTrainerTrainingPipeline,
)

STAGE_NAME = "Model Training"
try:
    print(f">>> Stage: {STAGE_NAME} <<<")
    pipeline_obj = ModelTrainerTrainingPipeline()
    pipeline_obj.initiate_model_trainer()
    print(f"✅ {STAGE_NAME} completed successfully.")
except Exception as e:
    print(f"❌ {STAGE_NAME} failed: {e})")
    raise

---
## 📊 Phase 5 — Stage 4: Model Evaluation

Computes ROUGE-1, ROUGE-2, ROUGE-L, and ROUGE-Lsum scores on the first 10
test examples.  Results are saved to `artifacts/model_evaluation/metrics.csv`.


In [ ]:
write_file("src/textSummarizer/components/model_evaluation.py", '''\
import os
import pandas as pd
from datasets import load_from_disk
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from evaluate import load

from src.textSummarizer.entity import ModelEvaluationConfig
from src.textSummarizer.logging import logger


class ModelEvaluation:
    """
    Evaluates the fine-tuned model using ROUGE metrics on the test dataset
    and saves the scores to a CSV file.
    """

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """Split the list into N chunks of size batch_size."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer, batch_size=16, device="cuda"):
        article_batches = list(self.generate_batch_sized_chunks(dataset["test"]["dialogue"], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset["test"]["summary"], batch_size))

        for article_batch, target_batch in zip(article_batches, target_batches):
            inputs = tokenizer(article_batch, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")
            with torch.no_grad():
                summaries = model.generate(
                    input_ids=inputs["input_ids"].to(device),
                    attention_mask=inputs["attention_mask"].to(device),
                    length_penalty=0.8,
                    num_beams=8,
                    max_length=128
                )
            decoded_summaries = tokenizer.batch_decode(summaries, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]
            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        score = metric.compute()
        return score

    def evaluate(self) -> dict:
        device = "cuda" if torch.cuda.is_available() else "cpu"

        logger.info(f"Loading model from {self.config.model_path} and tokenizer from {self.config.tokenizer_path}")
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)

        logger.info(f"Loading dataset from {self.config.data_path}")
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # Load ROUGE metric
        rouge_metric = load("rouge")

        logger.info("Calculating ROUGE metrics on test dataset (this may take a while) …")
        score = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum_pt,
            metric=rouge_metric,
            model=model_pegasus,
            tokenizer=tokenizer,
            batch_size=2, # Reduced batch size to prevent OOM errors
            device=device,
        )

        rouge_scores = {
            "rouge1": score["rouge1"].mid.fmeasure,
            "rouge2": score["rouge2"].mid.fmeasure,
            "rougeL": score["rougeL"].mid.fmeasure,
            "rougeLsum": score["rougeLsum"].mid.fmeasure,
        }

        # Save scores to CSV
        pd.DataFrame(rouge_scores, index=[0]).to_csv(self.config.metric_file_name, index=False)
        logger.info(f"ROUGE scores saved to {self.config.metric_file_name}")

        return rouge_scores
''')
print("✅ ModelEvaluation component written.")

# Reload the component module first
import importlib
from src.textSummarizer.components import model_evaluation
importlib.reload(model_evaluation)
# Then reload the pipeline module that uses the component
from src.textSummarizer.pipeline import stage_4_model_evaluation
importlib.reload(stage_4_model_evaluation)

from src.textSummarizer.pipeline.stage_4_model_evaluation import (
    ModelEvaluationTrainingPipeline,
)

STAGE_NAME = "Model Evaluation"
try:
    print(f">>> Stage: {STAGE_NAME} <<<")
    pipeline_obj = ModelEvaluationTrainingPipeline()
    scores = pipeline_obj.initiate_model_evaluation()
    print(f"\n✅ {STAGE_NAME} completed successfully.")
    print("\nROUGE Scores:")
    for k, v in scores.items():
        print(f"  {k}: {v:.4f}")
except Exception as e:
    print(f"❌ {STAGE_NAME} failed: {e}")
    raise

In [ ]:
write_file("src/textSummarizer/components/model_evaluation.py", '''\
import os
import pandas as pd
from datasets import load_from_disk
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from evaluate import load

from src.textSummarizer.entity import ModelEvaluationConfig
from src.textSummarizer.logging import logger


class ModelEvaluation:
    """
    Evaluates the fine-tuned model using ROUGE metrics on the test dataset
    and saves the scores to a CSV file.
    """

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer, batch_size=16, device="cuda"):

        # 🔥 LIMIT DATA FOR SPEED (huge improvement)
        dataset = dataset["test"].select(range(50))   # change 50 → 100 if needed

        article_batches = list(self.generate_batch_sized_chunks(dataset["dialogue"], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset["summary"], batch_size))

        model.eval()

        for article_batch, target_batch in zip(article_batches, target_batches):

            inputs = tokenizer(
                article_batch,
                truncation=True,
                padding=True,   # 🔥 dynamic padding (faster)
                return_tensors="pt"
            ).to(device)

            with torch.no_grad():
                # 🔥 mixed precision (faster on GPU)
                with torch.cuda.amp.autocast():
                    summaries = model.generate(
                        input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"],
                        length_penalty=0.8,
                        num_beams=4,      # 🔥 reduced from 8 → faster
                        max_length=128
                    )

            decoded_summaries = tokenizer.batch_decode(
                summaries,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )

            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        return metric.compute()

    def evaluate(self) -> dict:
        device = "cuda" if torch.cuda.is_available() else "cpu"

        logger.info(f"Loading model from {self.config.model_path}")
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)

        logger.info(f"Loading dataset from {self.config.data_path}")
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        rouge_metric = load("rouge")

        logger.info("Calculating ROUGE metrics (FAST MODE) …")
        score = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum_pt,
            metric=rouge_metric,
            model=model_pegasus,
            tokenizer=tokenizer,
            batch_size=4,   # 🔥 increased batch size (GPU efficient)
            device=device,
        )

        rouge_scores = {
            "rouge1": score["rouge1"],
            "rouge2": score["rouge2"],
            "rougeL": score["rougeL"],
            "rougeLsum": score["rougeLsum"],
        }

        pd.DataFrame(rouge_scores, index=[0]).to_csv(self.config.metric_file_name, index=False)
        logger.info(f"ROUGE scores saved to {self.config.metric_file_name}")

        return rouge_scores
''')

print("✅ FAST ModelEvaluation written.")

### Re-run Stage 4: Model Evaluation (with FAST optimizations)

In [ ]:
# ── Run Stage 4: Model Evaluation ───────────────────────────────────────────────
import importlib
from src.textSummarizer.components import model_evaluation
from src.textSummarizer.pipeline import stage_4_model_evaluation

# Reload the component and pipeline modules to pick up changes from zR6Vh2kToCI_
importlib.reload(model_evaluation)
importlib.reload(stage_4_model_evaluation)

from src.textSummarizer.pipeline.stage_4_model_evaluation import (
    ModelEvaluationTrainingPipeline,
)

STAGE_NAME = "Model Evaluation"
try:
    print(f">>> Stage: {STAGE_NAME} <<<")
    pipeline_obj = ModelEvaluationTrainingPipeline()
    scores = pipeline_obj.initiate_model_evaluation()
    print(f"\n✅ {STAGE_NAME} completed successfully.")
    print("\nROUGE Scores:")
    for k, v in scores.items():
        print(f"  {k}: {v:.4f}")
except Exception as e:
    print(f"❌ {STAGE_NAME} failed: {e}")
    raise

In [ ]:
# ── Display metrics CSV as a DataFrame ────────────────────────────────────────
import pandas as pd

metrics_path = "artifacts/model_evaluation/metrics.csv"
if os.path.exists(metrics_path):
    df = pd.read_csv(metrics_path)
    print("ROUGE Evaluation Results:")
    display(df)
else:
    print("metrics.csv not found — did the evaluation stage complete?")


---
## 🤖 Phase 6 — Prediction Demo (Fine-tuned PEGASUS)

Run inference on a custom dialogue using the fine-tuned model.


In [ ]:
from src.textSummarizer.pipeline.prediction_pipeline import PredictionPipeline

pipeline_pred = PredictionPipeline()
summary = pipeline_pred.predict(sample_dialogue)

print(summary)

In [ ]:
# ── Predict with fine-tuned PEGASUS ──────────────────────────────────────────
from src.textSummarizer.pipeline.prediction_pipeline import PredictionPipeline

# Sample dialogue from SAMSum-style data
sample_dialogue = """\
Hannah: Hey, are you coming to the party tonight?
Eric: I'm not sure, I have to finish this project first.
Hannah: It starts at 9, you should be done by then!
Eric: Okay, but only for an hour. I really need to submit this by midnight.
Hannah: Deal! I'll save you a slice of cake.
Eric: Cake? Now I'm definitely coming.
"""

print("=== Input Dialogue ===")
print(sample_dialogue)

# Attempt to upgrade transformers to ensure summarization task is available
import subprocess
import sys
import importlib
print("Attempting to upgrade transformers library...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "transformers", "-q"])
print("Transformers upgrade complete. Reloading transformers module.")

# Reload the transformers module to ensure the updated task registry is active
import transformers
importlib.reload(transformers)

# Re-import PredictionPipeline after upgrade and module reload
from src.textSummarizer.pipeline.prediction_pipeline import PredictionPipeline

pipeline_pred = PredictionPipeline()
summary = pipeline_pred.predict(sample_dialogue)

print("\n=== Generated Summary ===")
print(summary)


---
## ⚡ Phase 7 — Groq LLM Demo (llama-3.1-8b-instant)

This phase uses **Groq's ultra-fast inference API** with the
`llama-3.1-8b-instant` model to summarise dialogues without any local GPU usage.

**Setup:**
1. Get a free API key at [console.groq.com](https://console.groq.com)
2. In Colab: left sidebar → 🔑 Secrets → Add `GROQ_API_KEY`

The secret is accessed via `google.colab.userdata` — it never appears in the
notebook output and is safe to push to Git.


In [ ]:
# ── Load GROQ_API_KEY from Colab Secrets ──────────────────────────────────────
# Colab Secrets is the safe way to store API keys — they are never
# embedded in the notebook file and do not appear in outputs.

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded from Colab Secrets.")
except Exception:
    # Fallback for local Jupyter / non-Colab environments
    import os
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
    if GROQ_API_KEY:
        print("✅ GROQ_API_KEY loaded from environment variable.")
    else:
        print("⚠️  GROQ_API_KEY not found. Add it to Colab Secrets (🔑 in the sidebar).")


In [ ]:
# ── Groq summarisation function ───────────────────────────────────────────────
# Uses the groq Python SDK (v0.x).  The llama-3.1-8b-instant model is optimised
# for speed; for higher quality switch to llama-3.3-70b-versatile.

from groq import Groq

def summarise_with_groq(dialogue: str, api_key: str) -> str:
    """
    Send a dialogue to Groq's llama-3.1-8b-instant model and return
    a concise summary.

    Args:
        dialogue (str): Raw conversation text to summarise.
        api_key (str): Groq API key.

    Returns:
        str: Model-generated summary.
    """
    client = Groq(api_key=api_key)

    # System prompt instructs the model to act as a summarisation assistant
    system_prompt = (
        "You are a helpful assistant that summarises conversations concisely. "
        "Given a dialogue, produce a 1-3 sentence summary capturing the key points."
    )

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",   # Groq's fast LLaMA 3.1 8B endpoint
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Summarise the following dialogue:\n\n{dialogue}"},
        ],
        temperature=0.3,     # low temperature for factual summaries
        max_tokens=256,
    )

    return response.choices[0].message.content


In [ ]:
# ── Run Groq summarisation ────────────────────────────────────────────────────
if GROQ_API_KEY:
    groq_summary = summarise_with_groq(sample_dialogue, GROQ_API_KEY)
    print("=== Input Dialogue ===")
    print(sample_dialogue)
    print("\n=== Groq (llama-3.1-8b-instant) Summary ===")
    print(groq_summary)
else:
    print("⚠️  Skipping Groq demo — GROQ_API_KEY not set.")


In [ ]:
# ── Batch Groq summarisation on SAMSum test examples ─────────────────────────
# Shows how to run Groq inference over the actual dataset for comparison.

if GROQ_API_KEY:
    from datasets import load_from_disk

    dataset = load_from_disk("artifacts/data_ingestion/samsum_dataset")
    test_samples = dataset["test"].select(range(3))   # first 3 test examples

    print("Comparing Reference vs Groq Summaries\n" + "=" * 50)
    for i, sample in enumerate(test_samples):
        groq_sum = summarise_with_groq(sample["dialogue"], GROQ_API_KEY)
        print(f"\n--- Example {i+1} ---")
        print(f"Reference : {sample['summary']}")
        print(f"Groq      : {groq_sum}")
else:
    print("⚠️  Skipping batch demo — GROQ_API_KEY not set.")


---
## 🚀 Phase 8 — App Launch (FastAPI + ngrok)

Launches the FastAPI inference server inside Colab and exposes it publicly
via **ngrok** so you can call `/predict` from anywhere.

**Endpoints:**
- `GET  /`        → Redirects to `/docs` (Swagger UI)
- `GET  /train`   → Triggers the full training pipeline
- `POST /predict` → Summarises a given text

> 📌 Set your `NGROK_AUTHTOKEN` in Colab Secrets to use this section.  
> Get a free token at [dashboard.ngrok.com](https://dashboard.ngrok.com).


In [ ]:
# ── Install ngrok ─────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "fastapi", "uvicorn", "pyngrok"])
print("✅ FastAPI, uvicorn, pyngrok installed.")


In [ ]:
# ── Write app.py ──────────────────────────────────────────────────────────────
write_file("app.py", """\
import os
import sys

# Add project root to path so src.textSummarizer.* imports work
sys.path.insert(0, "/content/TextSummarizer")
os.chdir("/content/TextSummarizer")

from fastapi import FastAPI
from fastapi.responses import Response
from starlette.responses import RedirectResponse
import uvicorn

from src.textSummarizer.pipeline.prediction_pipeline import PredictionPipeline

app = FastAPI(title="Text Summarizer API", version="1.0.0")


@app.get("/", tags=["Navigation"])
async def index():
    """Redirect root to the interactive Swagger docs."""
    return RedirectResponse(url="/docs")


@app.get("/train", tags=["Training"])
async def training():
    """
    Trigger the full training pipeline (all 4 stages).
    Runs synchronously — may take 30-60 minutes on a T4 GPU.
    """
    try:
        from src.textSummarizer.pipeline.stage_1_data_ingestion_pipeline import DataIngestionTrainingPipeline
        from src.textSummarizer.pipeline.stage_2_data_transformation_pipeline import DataTransformationTrainingPipeline
        from src.textSummarizer.pipeline.stage_3_model_trainer_pipeline import ModelTrainerTrainingPipeline
        from src.textSummarizer.pipeline.stage_4_model_evaluation import ModelEvaluationTrainingPipeline

        DataIngestionTrainingPipeline().initiate_data_ingestion()
        DataTransformationTrainingPipeline().initiate_data_transformation()
        ModelTrainerTrainingPipeline().initiate_model_trainer()
        ModelEvaluationTrainingPipeline().initiate_model_evaluation()
        return Response("Training pipeline completed successfully!")
    except Exception as e:
        return Response(f"Training failed: {e}", status_code=500)


@app.post("/predict", tags=["Inference"])
async def predict_route(text: str):
    """
    Summarise the provided text using the fine-tuned PEGASUS model.

    Args:
        text (str): Dialogue or article to summarise.

    Returns:
        str: Generated summary.
    """
    try:
        predictor = PredictionPipeline()
        summary = predictor.predict(text)
        return {"summary": summary}
    except Exception as e:
        return Response(f"Prediction failed: {e}", status_code=500)


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8080)
""")
print("✅ app.py written.")


In [ ]:
# ── Launch FastAPI with ngrok tunnel ─────────────────────────────────────────
# The server runs in a background thread so the notebook stays interactive.
# The public ngrok URL is printed below — open it in your browser.

import threading
import uvicorn
from pyngrok import ngrok, conf

# Load ngrok token from Colab Secrets (optional but recommended)
try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN")
    if ngrok_token:
        conf.get_default().auth_token = ngrok_token
        print("✅ ngrok auth token set.")
except Exception:
    print("ℹ️  NGROK_AUTHTOKEN not found — public URL may require authentication.")

# Start the FastAPI server in a daemon thread
def run_server():
    """Run uvicorn server; daemon=True means it stops when the notebook stops."""
    import importlib.util, sys
    spec = importlib.util.spec_from_file_location("app", "/content/TextSummarizer/app.py")
    app_module = importlib.util.load_from_spec(spec)
    spec.loader.exec_module(app_module)
    uvicorn.run(app_module.app, host="0.0.0.0", port=8080, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(3)  # wait for uvicorn to boot

# Open an ngrok tunnel to expose port 8080 publicly
public_url = ngrok.connect(8080)
print(f"\n🌐 Public URL : {public_url}")
print(f"   Swagger UI  : {public_url}/docs")
print(f"   Predict     : POST {public_url}/predict?text=<your+dialogue>")


---
## 📦 Phase 9 — Package All Outputs

Zips the `artifacts/` directory (model weights, tokenizer, metrics CSV, datasets)
and downloads it to your local machine.


In [ ]:
# ── Zip all pipeline artifacts ────────────────────────────────────────────────
import zipfile
import os
from pathlib import Path

OUTPUT_ZIP = "/content/text_summarizer_outputs.zip"

def zip_directory(folder_path: str, output_path: str):
    """
    Recursively zip an entire directory.

    Args:
        folder_path: Root directory to zip.
        output_path: Destination .zip file path.
    """
    with zipfile.ZipFile(output_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                full_path = os.path.join(root, file)
                # Store with a relative arcname so unzip preserves structure
                arcname = os.path.relpath(full_path, start=os.path.dirname(folder_path))
                zf.write(full_path, arcname)
    size_mb = Path(output_path).stat().st_size / 1e6
    print(f"✅ Zipped → {output_path}  ({size_mb:.1f} MB)")

zip_directory("/content/TextSummarizer/artifacts", OUTPUT_ZIP)


In [ ]:
# ── Also zip the logs directory ───────────────────────────────────────────────
LOGS_ZIP = "/content/text_summarizer_logs.zip"
logs_path = "/content/TextSummarizer/logs"

if os.path.exists(logs_path):
    zip_directory(logs_path, LOGS_ZIP)
else:
    print("No logs directory found — skipping logs zip.")


In [ ]:
# ── Download zips to local machine ────────────────────────────────────────────
# google.colab.files.download triggers a browser download dialog.

try:
    from google.colab import files

    print("Downloading artifacts zip …")
    files.download(OUTPUT_ZIP)

    if os.path.exists(LOGS_ZIP):
        print("Downloading logs zip …")
        files.download(LOGS_ZIP)

    print("✅ Downloads initiated — check your browser's download folder.")
except ImportError:
    print("Not running in Colab — find outputs at:")
    print(f"  {OUTPUT_ZIP}")
    print(f"  {LOGS_ZIP}")


---
## ✅ Pipeline Complete!

| Phase | Status |
|-------|--------|
| 0 — Environment Setup | ✅ |
| 1 — Project Structure | ✅ |
| 2 — Data Ingestion | ✅ |
| 3 — Data Transformation | ✅ |
| 4 — Model Training | ✅ |
| 5 — Model Evaluation | ✅ |
| 6 — Prediction Demo | ✅ |
| 7 — Groq LLM Demo | ✅ |
| 8 — App Launch | ✅ |
| 9 — Output Packaging | ✅ |

**Generated artifacts:**
- `artifacts/data_ingestion/` — raw + extracted dataset  
- `artifacts/data_transformation/` — tokenized Arrow dataset  
- `artifacts/model_trainer/` — fine-tuned PEGASUS weights + tokenizer  
- `artifacts/model_evaluation/metrics.csv` — ROUGE scores  
- `/content/text_summarizer_outputs.zip` — everything zipped for download
